In [0]:
import uuid
from datetime import datetime, timezone
from delta.tables import DeltaTable
from pyspark.sql import Window  # type: ignore
from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, StringType, StructField, StructType

dbutils.widgets.text("catalog_name", "weather_cat")
CATALOG = dbutils.widgets.get("catalog_name")

spark.sql(f"USE CATALOG {CATALOG}")

# Fetch the latest run from Bronze as a fallback for interactive or partial (--only) runs
latest_run = spark.sql(f"""
    SELECT run_id,
           MIN(ingestion_ts) AS start_ts,
           COUNT(*) AS attempted,
           SUM(CASE WHEN http_status = 200 THEN 1 ELSE 0 END) AS succeeded
    FROM {CATALOG}.bronze.weather_raw
    GROUP BY run_id
    ORDER BY start_ts DESC
    LIMIT 1
""").first()

if not latest_run:
    raise RuntimeError("No records found in bronze.weather_raw. Run 02_ingest_bronze first!")

fallback_run_id = latest_run["run_id"]

# Retrieve Workflow taskValues with BOTH default and debugValue fallbacks
RUN_ID = dbutils.jobs.taskValues.get(
    taskKey="ingest_bronze",
    key="run_id",
    default=fallback_run_id,
    debugValue=fallback_run_id,
)

bronze_df = spark.table(f"{CATALOG}.bronze.weather_raw").filter(
    (F.col("run_id") == RUN_ID) & (F.col("http_status") == 200)
)

In [0]:
# Parse hourly arrays as strings first so changed types (e.g. "25.4" as string) don't crash from_json
hourly_schema = StructType(
    [
        StructField(
            "hourly",
            StructType(
                [
                    StructField("time", ArrayType(StringType()), True),
                    StructField("temperature_2m", ArrayType(StringType()), True),
                    StructField(
                        "relative_humidity_2m", ArrayType(StringType()), True
                    ),
                    StructField("precipitation", ArrayType(StringType()), True),
                    StructField("wind_speed_10m", ArrayType(StringType()), True),
                ]
            ),
            True,
        )
    ]
)

parsed_df = (
    bronze_df.withColumn("json_data", F.from_json("raw_response", hourly_schema))
    .filter("json_data.hourly IS NOT NULL")
    .withColumn(
        "zipped",
        F.arrays_zip(
            "json_data.hourly.time",
            "json_data.hourly.temperature_2m",
            "json_data.hourly.relative_humidity_2m",
            "json_data.hourly.precipitation",
            "json_data.hourly.wind_speed_10m",
        ),
    )
    .select("location_id", "city_name", F.explode("zipped").alias("row"))
)

In [0]:
# Safe type casting (handles numbers returned as strings; invalid strings become NULL)
typed_df = parsed_df.select(
    F.col("location_id"),
    F.col("city_name"),
    F.col("row.time").alias("raw_time"),
    F.expr("try_cast(row.time as timestamp)").alias("forecast_ts"),
    F.expr("try_cast(row.temperature_2m as double)").alias("temperature_2m"),
    F.expr("try_cast(row.relative_humidity_2m as double)").alias(
        "relative_humidity_2m"
    ),
    F.expr("try_cast(row.precipitation as double)").alias("precipitation"),
    F.expr("try_cast(row.wind_speed_10m as double)").alias("wind_speed_10m"),
    F.to_json("row").alias("raw_record"),
)


In [0]:
# Data Quality Rules
validated_df = typed_df.withColumn(
    "quarantine_reason",
    F.when(F.col("forecast_ts").isNull(), "INVALID_OR_NULL_TIMESTAMP")
    .when(
        F.col("temperature_2m").isNull()
        | F.col("relative_humidity_2m").isNull()
        | F.col("precipitation").isNull()
        | F.col("wind_speed_10m").isNull(),
        "NULL_OR_NON_NUMERIC_METRIC",
    )
    .when(
        (F.col("temperature_2m") < -90) | (F.col("temperature_2m") > 60),
        "TEMPERATURE_OUT_OF_RANGE",
    )
    .when(
        (F.col("relative_humidity_2m") < 0)
        | (F.col("relative_humidity_2m") > 100),
        "HUMIDITY_OUT_OF_RANGE",
    )
    .when(F.col("precipitation") < 0, "NEGATIVE_PRECIPITATION")
    .when(
        (F.col("wind_speed_10m") < 0) | (F.col("wind_speed_10m") > 400),
        "WIND_SPEED_OUT_OF_RANGE",
    )
    .otherwise(F.lit(None)),
)

In [0]:
# Split into Invalid (Quarantine) and Valid (Silver)
invalid_df = validated_df.filter("quarantine_reason IS NOT NULL").select(
    F.expr("uuid()").alias("quarantine_id"),
    F.lit(RUN_ID).alias("run_id"),
    F.lit("SILVER_VALIDATION").alias("stage"),
    F.col("location_id"),
    F.col("raw_time").alias("forecast_ts"),
    F.col("quarantine_reason"),
    F.col("raw_record"),
    F.current_timestamp().alias("quarantined_at"),
)

if invalid_df.count() > 0:
    invalid_df.write.mode("append").saveAsTable(
        f"{CATALOG}.quarantine.weather_quarantine"
    )


In [0]:
# Deduplicate valid rows on (location_id, forecast_ts) in case API returns duplicate timestamps
window_spec = Window.partitionBy("location_id", "forecast_ts").orderBy(
    F.col("forecast_ts").desc()
)
valid_silver_df = (
    validated_df.filter("quarantine_reason IS NULL")
    .withColumn("rn", F.row_number().over(window_spec))
    .filter("rn = 1")
    .select(
        "location_id",
        "city_name",
        "forecast_ts",
        "temperature_2m",
        "relative_humidity_2m",
        "precipitation",
        "wind_speed_10m",
        F.lit(RUN_ID).alias("run_id"),
        F.current_timestamp().alias("updated_at"),
    )
)

In [0]:
# Idempotent MERGE into Silver (Re-running the same time window updates existing rows without duplicates)
silver_target = DeltaTable.forName(spark, f"{CATALOG}.silver.weather_hourly")
(
    silver_target.alias("t")
    .merge(
        valid_silver_df.alias("s"),
        "t.location_id = s.location_id AND t.forecast_ts = s.forecast_ts",
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

In [0]:
# Add at the very bottom of 03_transform_silver.ipynb (after MERGE and quarantine writes)
silver_count = spark.table(f"{CATALOG}.silver.weather_hourly").filter(F.col("run_id") == RUN_ID).count()
quarantine_count = spark.table(f"{CATALOG}.quarantine.weather_quarantine").filter(F.col("run_id") == RUN_ID).count()

dbutils.jobs.taskValues.set(key="silver_rows_merged", value=silver_count)
dbutils.jobs.taskValues.set(key="quarantined_rows", value=quarantine_count)

dbutils.notebook.exit(
    f"Silver Complete | Catalog: {CATALOG} | Run ID: {RUN_ID} | Merged: {silver_count} | Quarantined: {quarantine_count}"
)
